# AcademiQ - Notebook 05: Final Integration and Synthesis

This notebook integrates and synthesizes the complete simplified AcademiQ pipeline:
1. **EDA & Preprocessing Verification:** Confirms dataset integrity and preparation pipeline.
2. **Regression Benchmark Comparison:** Compares Simple Linear Regression, Multiple Linear Regression, and Polynomial Regression.
3. **Model Selection Defense:** Establishes why Multiple Linear Regression is crowned champion ($R^2 = 0.7699$, $\text{MAE} = 0.4498$).
4. **Student Factor Analysis Synthesis:** Connects univariate feature selection to regression findings.
5. **Live Score Prediction & Threshold Categorization:** Demonstrates pipeline inference and threshold-based academic tier assignment.
6. **What-If Academic Scenario Analysis:** Demonstrates controlled scenario comparisons with explicit non-causal disclaimers.

In [ ]:
import os
from pathlib import Path

def get_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / "data" / "raw").exists() or (candidate / "requirements.txt").exists():
            return candidate
    return current

PROJECT_ROOT = get_project_root()
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

print(f"Project root resolved to: {PROJECT_ROOT}")
print("Libraries imported successfully.")

In [ ]:
# 1. Verify and Load Upstream Analytical Results
res_dir = PROJECT_ROOT / "results"

regression_test_file = res_dir / "regression" / "AcademiQ_Regression_Test_Results.csv"
regression_cv_file = res_dir / "regression" / "AcademiQ_Regression_CV_Results.csv"
feature_selection_file = res_dir / "feature_analysis" / "AcademiQ_Feature_Selection_Results.csv"
selected_features_file = res_dir / "feature_analysis" / "AcademiQ_Selected_Features.csv"

regression_results = pd.read_csv(regression_test_file)
regression_cv = pd.read_csv(regression_cv_file)
feature_scores = pd.read_csv(feature_selection_file)
selected_features = pd.read_csv(selected_features_file)

print("=== Regression Test Results ===")
display(regression_results)

print("\n=== Regression 5-Fold Cross-Validation Results ===")
display(regression_cv)

In [ ]:
# 2. Identify and Defend Champion Regression Model
best_regression = regression_results.loc[regression_results["R2 Score"].idxmax()]

print("Champion Regression Model Selection:")
print(f"  Model:          {best_regression['Model']}")
print(f"  Test R2 Score:  {best_regression['R2 Score']:.4f}")
print(f"  Test MAE:       {best_regression['MAE']:.4f}")
print(f"  Test RMSE:      {best_regression['RMSE']:.4f}")

print("\nAcademic Defense of Model Selection:")
print("  - Simple Linear Regression explains only 23.2% of score variance (severe underfitting).")
print("  - Multiple Linear Regression explains 77.0% of score variance with MAE = 0.45 points.")
print("  - Polynomial Regression (degree 2) achieves R2 = 76.8% with MAE = 0.47 points, offering no")
print("    practical advantage while adding 500+ interaction parameters. MLR satisfies Occam's razor.")

In [ ]:
# 3. Student Factor Analysis Synthesis
print("Top 10 Student Factors Associated with Exam_Score (SelectKBest ANOVA F-test):")
display(feature_scores.head(10))

dominant = feature_scores.iloc[:2]
print("\nDominant Predictors:")
for _, row in dominant.iterrows():
    print(f"  - {row['Feature']}: F-Score = {row['F_Score']:.2f} (p-value = {row['P_Value']:.2e})")

In [ ]:
# 4. Generate Consolidated Final Model Summary Table
final_summary = pd.DataFrame([
    {
        "Model": "Simple Linear Regression",
        "Input_Scope": "Hours_Studied Only",
        "Test_R2": regression_results.loc[regression_results["Model"] == "Simple Linear Regression", "R2 Score"].values[0],
        "Test_MAE": regression_results.loc[regression_results["Model"] == "Simple Linear Regression", "MAE"].values[0],
        "Test_RMSE": regression_results.loc[regression_results["Model"] == "Simple Linear Regression", "RMSE"].values[0],
        "Selected_Status": "Baseline (Underfitting)"
    },
    {
        "Model": "Multiple Linear Regression",
        "Input_Scope": "All 19 Student Factors",
        "Test_R2": regression_results.loc[regression_results["Model"] == "Multiple Linear Regression", "R2 Score"].values[0],
        "Test_MAE": regression_results.loc[regression_results["Model"] == "Multiple Linear Regression", "MAE"].values[0],
        "Test_RMSE": regression_results.loc[regression_results["Model"] == "Multiple Linear Regression", "RMSE"].values[0],
        "Selected_Status": "CHAMPION MODEL (Optimal Parity & Interpretability)"
    },
    {
        "Model": "Polynomial Regression (Degree 2)",
        "Input_Scope": "All 19 Factors + Degree-2 Interactions",
        "Test_R2": regression_results.loc[regression_results["Model"] == "Polynomial Regression", "R2 Score"].values[0],
        "Test_MAE": regression_results.loc[regression_results["Model"] == "Polynomial Regression", "MAE"].values[0],
        "Test_RMSE": regression_results.loc[regression_results["Model"] == "Polynomial Regression", "RMSE"].values[0],
        "Selected_Status": "Evaluated (Capacity Plateau / Overfitting Risk)"
    }
])

results_final_dir = PROJECT_ROOT / "results" / "final"
results_final_dir.mkdir(parents=True, exist_ok=True)

final_summary.to_csv(results_final_dir / "AcademiQ_Final_Model_Summary.csv", index=False)
print("Master Model Summary saved to results/final/AcademiQ_Final_Model_Summary.csv:")
display(final_summary)

In [ ]:
# 5. Generate and Save Comparative Regression Visualization Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = regression_results["Model"].str.replace(" ", "\n")
colors = ["#d9534f", "#2b5c8f", "#6c757d"]

# R2 Comparison
axes[0].bar(models, regression_results["R2 Score"], color=colors, edgecolor="black", alpha=0.85)
axes[0].set_title("Goodness-of-Fit (R2 Score) \n[Higher is Better]", fontsize=11, fontweight="bold")
axes[0].set_ylabel("R2 Score", fontsize=10)
axes[0].set_ylim(0, 1.0)
for i, v in enumerate(regression_results["R2 Score"]):
    axes[0].text(i, v + 0.02, f"{v:.4f}", ha="center", fontweight="bold")

# MAE Comparison
axes[1].bar(models, regression_results["MAE"], color=colors, edgecolor="black", alpha=0.85)
axes[1].set_title("Mean Absolute Error (MAE) \n[Lower is Better]", fontsize=11, fontweight="bold")
axes[1].set_ylabel("MAE (Exam Score Points)", fontsize=10)
axes[1].set_ylim(0, 3.0)
for i, v in enumerate(regression_results["MAE"]):
    axes[1].text(i, v + 0.05, f"{v:.4f}", ha="center", fontweight="bold")

# RMSE Comparison
axes[2].bar(models, regression_results["RMSE"], color=colors, edgecolor="black", alpha=0.85)
axes[2].set_title("Root Mean Squared Error (RMSE) \n[Lower is Better]", fontsize=11, fontweight="bold")
axes[2].set_ylabel("RMSE (Exam Score Points)", fontsize=10)
axes[2].set_ylim(0, 4.0)
for i, v in enumerate(regression_results["RMSE"]):
    axes[2].text(i, v + 0.08, f"{v:.4f}", ha="center", fontweight="bold")

plt.suptitle("AcademiQ Regression Benchmark Comparison", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(results_final_dir / "AcademiQ_Regression_Comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {results_final_dir / 'AcademiQ_Regression_Comparison.png'}")

In [ ]:
# 6. Live Prediction and Threshold-Based Performance Categorization
model_path = PROJECT_ROOT / "models" / "AcademiQ_Multiple_Linear_Regression.pkl"
pipeline = joblib.load(model_path)

# Define sample student profile
sample_student = pd.DataFrame([{
    "Hours_Studied": 22,
    "Attendance": 88,
    "Parental_Involvement": "High",
    "Access_to_Resources": "High",
    "Extracurricular_Activities": "Yes",
    "Sleep_Hours": 7,
    "Previous_Scores": 80,
    "Motivation_Level": "High",
    "Internet_Access": "Yes",
    "Tutoring_Sessions": 2,
    "Family_Income": "Medium",
    "Teacher_Quality": "High",
    "School_Type": "Public",
    "Peer_Influence": "Positive",
    "Physical_Activity": 3,
    "Learning_Disabilities": "No",
    "Parental_Education_Level": "College",
    "Distance_from_Home": "Near",
    "Gender": "Female"
}])

predicted_score = pipeline.predict(sample_student)[0]

def assign_performance_category(score):
    if score <= 66.0:
        return "Developing", "Students who may benefit from academic support or targeted interventions."
    elif score <= 69.0:
        return "Proficient", "Students performing at grade-level expectations."
    else:
        return "Advanced", "Students demonstrating strong academic mastery."

category, description = assign_performance_category(predicted_score)

print("Sample Student Inference Demo:")
print(f"  Predicted Exam_Score: {predicted_score:.2f} points")
print(f"  Assigned Tier:        {category}")
print(f"  Tier Description:     {description}")
print("\n  * Note: Category is assigned using transparent threshold logic without a separate classification model.")

In [ ]:
# 7. What-If Scenario Analysis Demonstration
print("What-If Academic Scenario Analysis Demo:")
print("  Exploring the effect of altering Attendance while holding all other 18 inputs constant:")

scenario_attendance_values = [65, 75, 85, 95]
base_profile = sample_student.copy()

scenario_results = []
for att in scenario_attendance_values:
    temp_profile = base_profile.copy()
    temp_profile["Attendance"] = att
    pred = pipeline.predict(temp_profile)[0]
    cat, _ = assign_performance_category(pred)
    scenario_results.append({
        "Scenario_Attendance": f"{att}%",
        "Predicted_Score": round(pred, 2),
        "Score_Difference": round(pred - predicted_score, 2),
        "Performance_Category": cat
    })

display(pd.DataFrame(scenario_results))

print("\n" + "="*65)
print("CRITICAL NON-CAUSAL DISCLAIMER:")
print("The score difference is a statistical model-based prediction estimate.")
print("It reflects historical correlations in the training data, NOT a guaranteed")
print("real-world score improvement. Observational data cannot establish causality.")
print("="*65)

## Final Synthesis Conclusion

AcademiQ demonstrates that student academic performance can be accurately modeled using parsimonious linear machine learning ($R^2 = 0.77$, $\text{MAE} = 0.45$). By combining Multiple Linear Regression with statistical feature analysis, transparent performance categorization, and what-if scenario exploration, the project provides an interpretable and academically grounded tool for educational research.